# 🗂️ Notebook 2: Data Model & APIs

Now that we have an architecture, let's design the **boundary**: the data types and HTTP endpoints the rest of the world sees.

We will:
1. Model the entities (User, ConnectionRequest, Edge) with **pydantic**.
2. Walk through the **connection-request state machine** and validate it in code.
3. Sketch REST endpoints and implement an **in-process service** that passes small tests.
4. Use **SQLite** as a stand-in for the sharded edge DB from notebook 1.

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Then select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab has **no external services** (no Postgres, no Redis). Everything runs in-process with Python stdlib + `pydantic`, so you can focus on the concepts.

## 1. Entities

- **User** — a profile. We keep this minimal; the connections service does not care about titles or avatars.
- **ConnectionRequest** — a directed, stateful message: `pending → accepted | rejected | withdrawn`.
- **Edge** — an accepted, undirected relationship. Created **only** when a request is accepted.

Keeping requests and edges as *different* entities avoids a common bug: showing pending requests as if they were real connections.

In [ ]:
from datetime import datetime
from typing  import Literal, Optional
from pydantic import BaseModel, Field, field_validator

RequestStatus = Literal['pending', 'accepted', 'rejected', 'withdrawn']

class User(BaseModel):
    id:   int
    name: str

class ConnectionRequest(BaseModel):
    id:         int
    from_user:  int
    to_user:    int
    status:     RequestStatus = 'pending'
    created_at: datetime = Field(default_factory=datetime.utcnow)

    @field_validator('to_user')
    @classmethod
    def no_self_connect(cls, v, info):
        if v == info.data.get('from_user'):
            raise ValueError('cannot send a request to yourself')
        return v

class Edge(BaseModel):
    # Canonical ordering: a < b. Ensures each undirected edge has exactly one identity.
    a: int
    b: int
    since: datetime = Field(default_factory=datetime.utcnow)

    @field_validator('b')
    @classmethod
    def ordered(cls, v, info):
        a = info.data.get('a')
        if a is not None and a >= v:
            raise ValueError('edge must be stored with a < b (canonical order)')
        return v

# Quick check: pydantic rejects nonsense at the boundary.
try:
    ConnectionRequest(id=1, from_user=7, to_user=7)
except Exception as e:
    print('✅ validation caught:', e)

## 2. The request state machine

Most 'social graph' bugs live in this tiny state machine. Draw it once, encode it once, test it.

```
          send              accept
  (none) ──────► pending ─────────► accepted  (edge created)
                    │
                    │ reject            withdraw (by sender)
                    ▼                   │
                 rejected   ◄───────────┘
```

Rules encoded below:
- You cannot send a second request if one is already `pending` or `accepted`.
- Only the **recipient** can accept/reject. Only the **sender** can withdraw.
- Accepting a request creates **exactly one** canonical edge.

In [ ]:
from collections import defaultdict

class ConnectionsService:
    """Tiny in-memory service. Later we'll back it with SQLite / a real DB."""
    def __init__(self):
        self._requests: dict[int, ConnectionRequest] = {}
        self._edges:    set[tuple[int, int]] = set()        # canonical (a, b) with a < b
        self._adj:      dict[int, set[int]]  = defaultdict(set)  # mirrors _edges symmetrically
        self._next_id = 1

    # ---------- helpers ----------
    @staticmethod
    def _canon(u, v):
        return (u, v) if u < v else (v, u)

    def _already_linked(self, u, v):
        if self._canon(u, v) in self._edges:
            return True
        return any(
            r.status == 'pending'
            and {r.from_user, r.to_user} == {u, v}
            for r in self._requests.values()
        )

    # ---------- public API ----------
    def send_request(self, from_user: int, to_user: int) -> ConnectionRequest:
        if self._already_linked(from_user, to_user):
            raise ValueError('already connected or pending')
        req = ConnectionRequest(id=self._next_id, from_user=from_user, to_user=to_user)
        self._requests[req.id] = req
        self._next_id += 1
        return req

    def accept(self, request_id: int, as_user: int) -> Edge:
        req = self._requests[request_id]
        if req.to_user != as_user:     raise PermissionError('only recipient can accept')
        if req.status  != 'pending':   raise ValueError(f'cannot accept a {req.status} request')
        req.status = 'accepted'
        a, b = self._canon(req.from_user, req.to_user)
        self._edges.add((a, b))
        self._adj[a].add(b); self._adj[b].add(a)       # symmetric adjacency for O(1) lookup
        return Edge(a=a, b=b)

    def reject(self, request_id: int, as_user: int) -> ConnectionRequest:
        req = self._requests[request_id]
        if req.to_user != as_user:     raise PermissionError('only recipient can reject')
        if req.status  != 'pending':   raise ValueError(f'cannot reject a {req.status} request')
        req.status = 'rejected'
        return req

    def withdraw(self, request_id: int, as_user: int) -> ConnectionRequest:
        req = self._requests[request_id]
        if req.from_user != as_user:   raise PermissionError('only sender can withdraw')
        if req.status    != 'pending': raise ValueError(f'cannot withdraw a {req.status} request')
        req.status = 'withdrawn'
        return req

    def friends(self, user: int) -> list[int]:
        return sorted(self._adj[user])

svc = ConnectionsService()
svc.send_request(1, 2); svc.send_request(1, 3); svc.send_request(4, 1)
svc.accept(1, as_user=2)
svc.reject(2, as_user=3)
svc.accept(3, as_user=1)
print('friends of 1:', svc.friends(1))
print('friends of 2:', svc.friends(2))

### Let's prove the rules with tiny tests

If you change this service later, these asserts will scream if you break an invariant.

In [ ]:
def expect_raises(exc_type, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc_type as e:
        return e
    raise AssertionError(f'expected {exc_type.__name__}')

s = ConnectionsService()
r = s.send_request(1, 2)

# 1) only recipient can accept
expect_raises(PermissionError, s.accept, r.id, as_user=1)

# 2) cannot double-send while pending
expect_raises(ValueError, s.send_request, 1, 2)

# 3) happy path
s.accept(r.id, as_user=2)
assert s.friends(1) == [2] and s.friends(2) == [1]

# 4) once accepted, cannot re-connect
expect_raises(ValueError, s.send_request, 2, 1)

# 5) cannot accept an already-accepted request
expect_raises(ValueError, s.accept, r.id, as_user=2)

print('✅ all invariants hold')

## 3. HTTP API sketch

The service above is the *logic*. The HTTP surface is a thin adapter. A typical design:

| Method | Path | Body | What |
|---|---|---|---|
| `POST` | `/requests` | `{to_user}` | Current user sends a request |
| `POST` | `/requests/{id}/accept` | — | Recipient accepts |
| `POST` | `/requests/{id}/reject` | — | Recipient rejects |
| `DELETE` | `/requests/{id}` | — | Sender withdraws |
| `GET` | `/users/{id}/connections?cursor=…&limit=50` | — | 1st-degree list (paginated) |
| `GET` | `/users/{id}/degree/{other}` | — | Degree of separation (1/2/3 / 'far') |
| `GET` | `/users/{id}/pymk?limit=20` | — | People You May Know |

### Design notes
- **POST for state changes**, never GET — GETs must be idempotent and safe.
- Use **cursor pagination**, not `offset` — offsets drift when the list changes.
- Return **opaque ids** in URLs; never expose internal DB primary keys if they leak business info.
- **Idempotency keys** on `POST /requests` prevent duplicate requests when the client retries on a flaky network.

## 4. Persistence: the 'symmetric edges' table

Let's wire the service to SQLite using the pattern we chose in notebook 1 (two rows per edge).

In [ ]:
import sqlite3
con = sqlite3.connect(':memory:')
con.executescript('''
CREATE TABLE edges (
  owner  INTEGER NOT NULL,
  friend INTEGER NOT NULL,
  since  TEXT    NOT NULL,
  PRIMARY KEY (owner, friend)
) WITHOUT ROWID;
''')

def persist_accept(con, u, v):
    # In a real system these two inserts happen inside ONE transaction.
    # If the DB is sharded by 'owner', the two writes land on two shards, and we need
    # either a 2-phase commit or an outbox pattern. Here, a single transaction suffices.
    with con:
        con.execute('INSERT OR IGNORE INTO edges VALUES(?,?,datetime())', (u, v))
        con.execute('INSERT OR IGNORE INTO edges VALUES(?,?,datetime())', (v, u))

for a, b in [(1,2),(1,3),(2,3),(3,4),(4,5),(1,6)]:
    persist_accept(con, a, b)

def list_friends(con, user, cursor=0, limit=10):
    return [r[0] for r in con.execute(
        'SELECT friend FROM edges WHERE owner = ? AND friend > ? ORDER BY friend LIMIT ?',
        (user, cursor, limit))]

print('friends(1):', list_friends(con, 1))
print('friends(3):', list_friends(con, 3))
print('page2 of friends(1) after id=2:', list_friends(con, 1, cursor=2, limit=10))

### Why two writes, not one?

We keep the edge twice because the **access pattern** is always *'friends of this user'* — never *'find any edge touching this user in the whole table'*. Optimize for the query you run a million times a second.

The cost is extra storage and the need to keep the two rows in sync (transaction, or an outbox if shards differ). Every large social platform (Facebook TAO, Twitter FlockDB) makes this same trade.

## 5. Takeaways

- Model **requests** and **edges** as separate things — mixing them is bug factory #1.
- Encode the state machine in one place and guard it with a few tiny tests.
- HTTP is the **adapter**; the service object is the **logic**. Keep them separable.
- Store edges symmetrically, paginate with cursors, and wrap the two writes in a transaction.

Next: [`03_deep_dive.ipynb`](./03_deep_dive.ipynb) — BFS, bidirectional BFS, PYMK, and the 'celebrity' hot-user problem.